# T0.5 · Full texts

Ranks the included studies by priority, locates open-access copies, downloads them and leaves the list of what is still to be obtained.

- **Input:** `corpus.csv` of T0.2, `precoding_ai.csv` and `matrix.csv` of this folder, and `control_set.csv` of T0.1.
- **Output:** `full_texts.csv` (one study per row, with DOI, link, open copy and whether the PDF is already there), `priority_pending.txt` (DOI links of the priority ones still missing) and the `pdf/` folder.
- **PDFs:** they go in `pdf/` named after the identifier (`R1436.pdf`). The folder is not committed to the repository. Those obtained by hand are placed there under that same name.
- **Open access:** only legal copies are downloaded: the publisher's open version, or the one the author has deposited in a repository or preprint server.
- **To re-run:** run everything. It only queries the DOIs it has not yet recorded and only downloads the PDFs that are missing. It needs a connection.

In [1]:
import json, re, time, unicodedata, urllib.parse, urllib.request, warnings
warnings.filterwarnings("ignore")
from difflib import SequenceMatcher
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 90)
T02 = Path("../T0.2-search-and-deduplication")
CONTROL = Path("../T0.1-protocol/control_set.csv")
LIST = Path("full_texts.csv")
PDF = Path("pdf")
PDF.mkdir(exist_ok=True)
RETRY_FAILED = False      # True to try again the downloads that failed
AGENT = {"User-Agent": "CERTA-systematic-review/0.1 (academic use)"}

corpus = pd.read_csv(T02 / "corpus.csv", dtype=str).fillna("").set_index("id")
matrix = pd.read_csv("matrix.csv", dtype=str).fillna("").set_index("id")
pre = pd.read_csv("precoding_ai.csv", dtype=str).fillna("").set_index("id")
control = pd.read_csv(CONTROL, dtype=str).fillna("")

## 1. Reading priority

| Priority | Studies |
| --- | --- |
| 1 | Those that decide G0: two or three of the criteria C2, C3 and C4 in the pre-coding, or board level |
| 2 | Works of the control set (includes the four of the in-depth reading of T0.6) |
| 3 | The rest of the included studies |

In [2]:
C = ["C2_hw_cost", "C3_risk", "C4_certification"]
n_criteria = pre[C].ne("n").sum(axis=1).reindex(matrix.index).fillna(0)
board = pre["level"].reindex(matrix.index).eq("board")
in_control = corpus.loc[matrix.index, "doi"].isin(set(control["doi"].str.lower()) - {""})

lst = pd.DataFrame(index=matrix.index)
lst["priority"] = 3
lst.loc[in_control, "priority"] = 2
lst.loc[(n_criteria >= 2) | board, "priority"] = 1
lst["reason"] = ""
lst.loc[in_control, "reason"] = "control set"
lst.loc[board, "reason"] = "board level"
lst.loc[n_criteria >= 2, "reason"] = n_criteria[n_criteria >= 2].astype(int).astype(str) + " gap criteria"
lst["year"] = corpus.loc[lst.index, "year"]
lst["citation"] = matrix["citation"]
lst["title"] = corpus.loc[lst.index, "title"]
lst["doi"] = corpus.loc[lst.index, "doi"]
lst["link"] = lst["doi"].map(lambda d: "https://doi.org/" + d if d else "")

SAVED = ["oa", "oa_status", "urls", "sources", "download", "pdf_source", "tried_urls"]      # what is already known is kept
for col in SAVED:
    lst[col] = ""
if LIST.exists():
    previous = pd.read_csv(LIST, dtype=str).fillna("").set_index("id")
    common = lst.index.intersection(previous.index)
    for col in [c for c in SAVED if c in previous.columns]:
        lst.loc[common, col] = previous.loc[common, col]

lst["priority"].value_counts().sort_index().rename("studies").to_frame()

,studies
priority,
1,71
2,20
3,377


## 2. Where there are legal copies

Four sources, each queried only once per study (the `sources` column records which have been tried):

| Source | What it adds |
| --- | --- |
| OpenAlex by DOI | Open publisher version and copies in repositories |
| Semantic Scholar by DOI | Author copies that OpenAlex does not list |
| HAL by title | Manuscripts deposited by authors from French institutions |
| OpenAlex by title | Preprints with a different DOI (arXiv, Research Square, TechRxiv), only for priority 1 and 2 |

In [3]:
def read_json(url, data=None):
    headers = dict(AGENT, **({"Content-Type": "application/json"} if data else {}))
    req = urllib.request.Request(url, data=json.dumps(data).encode() if data else None, headers=headers)
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

def clean(t):
    return re.sub(r"[^a-z0-9]+", " ", unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower()).strip()

def similarity(a, b):
    return SequenceMatcher(None, clean(a), clean(b)).ratio()

candidates = {i: [u for u in lst.at[i, "urls"].split(" | ") if u] for i in lst.index}
def record(i, *urls):
    for u in urls:
        if u and u not in candidates[i]:
            candidates[i].append(u)
def missing(i, source):
    return source not in lst.at[i, "sources"].split(";")
def done(i, source):
    lst.at[i, "sources"] = ";".join(filter(None, lst.at[i, "sources"].split(";") + [source]))

with_doi = [i for i in lst.index if lst.at[i, "doi"]]
by_doi = {lst.at[i, "doi"]: i for i in with_doi}

# OpenAlex by DOI
pend = [i for i in with_doi if missing(i, "openalex")]
for k in range(0, len(pend), 40):
    batch = pend[k:k + 40]
    flt = "doi:" + "|".join(lst.at[i, "doi"] for i in batch)
    try:
        res = read_json("https://api.openalex.org/works?per-page=50&select=doi,open_access,locations&filter=" + urllib.parse.quote(flt, safe=":|/"))["results"]
    except Exception as e:
        print("OpenAlex, a batch failed:", type(e).__name__)
        continue
    for w in res:
        i = by_doi.get(w["doi"].lower().replace("https://doi.org/", ""))
        if i is None:
            continue
        oa = w.get("open_access") or {}
        lst.at[i, "oa"] = "y" if oa.get("is_oa") else "n"
        lst.at[i, "oa_status"] = oa.get("oa_status") or ""
        for loc in w.get("locations") or []:
            if loc.get("is_oa"):
                record(i, loc.get("pdf_url"), loc.get("landing_page_url"))
    for i in batch:
        done(i, "openalex")
    time.sleep(0.2)
print("OpenAlex by DOI, queried now:", len(pend))

# Semantic Scholar by DOI
pend = [i for i in with_doi if missing(i, "s2")]
for k in range(0, len(pend), 100):
    batch = pend[k:k + 100]
    for attempt in range(4):
        try:
            res = read_json("https://api.semanticscholar.org/graph/v1/paper/batch?fields=openAccessPdf",
                            {"ids": ["DOI:" + lst.at[i, "doi"] for i in batch]})
            for i, w in zip(batch, res):
                if w and (w.get("openAccessPdf") or {}).get("url"):
                    record(i, w["openAccessPdf"]["url"])
                done(i, "s2")
            break
        except Exception as e:
            time.sleep(5 * (attempt + 1))
    time.sleep(1.5)
print("Semantic Scholar, queried now:", len(pend))

OpenAlex by DOI, queried now: 0
Semantic Scholar, queried now: 0


In [4]:
has_pdf = lambda i: (PDF / f"{i}.pdf").exists()

# HAL by title
pend = [i for i in lst.index if missing(i, "hal") and not has_pdf(i)]
for i in pend:
    title = clean(lst.at[i, "title"])
    try:
        docs = read_json("https://api.archives-ouvertes.fr/search/?rows=3&fl=title_s,fileMain_s,uri_s&q=title_t:" + urllib.parse.quote('"' + title + '"'))["response"]["docs"]
        for d in docs:
            if d.get("fileMain_s") and max(similarity(t, lst.at[i, "title"]) for t in d.get("title_s", [""])) >= 0.9:
                record(i, d["fileMain_s"])
        done(i, "hal")
    except Exception as e:
        pass
    time.sleep(0.15)
print("HAL by title, queried now:", len(pend))

# OpenAlex by title: preprints with a different DOI
pend = [i for i in lst.index[lst["priority"] <= 2] if missing(i, "title") and not has_pdf(i) and not candidates[i]]
for i in pend:
    try:
        res = read_json("https://api.openalex.org/works?per-page=5&select=doi,title,locations&filter=open_access.is_oa:true,title.search:"
                        + urllib.parse.quote(clean(lst.at[i, "title"])))["results"]
        for w in res:
            if similarity(w.get("title") or "", lst.at[i, "title"]) >= 0.9:
                for loc in w.get("locations") or []:
                    if loc.get("is_oa"):
                        record(i, loc.get("pdf_url"), loc.get("landing_page_url"))
        done(i, "title")
    except Exception as e:
        pass
    time.sleep(0.2)
print("OpenAlex by title, queried now:", len(pend))

lst["urls"] = [" | ".join(candidates[i]) for i in lst.index]
lst["with_copy"] = lst["urls"].ne("").map({True: "y", False: "n"})
pd.crosstab(lst["priority"], lst["with_copy"].rename("with a copy located"), margins=True)

HAL by title, queried now: 0
OpenAlex by title, queried now: 0


with a copy located,n,y,All
priority,,,
1,67,4,71
2,15,5,20
3,330,47,377
All,412,56,468


## 3. Download

Tries each address located and keeps the first one that returns a PDF. If a repository page arrives, it looks in it for the link to the file (`citation_pdf_url`). Some sites reject automatic downloads; those are left as `failed` and have to be fetched by hand from the `urls` column.

In [5]:
BROWSER = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
           "Accept": "application/pdf,text/html;q=0.9,*/*;q=0.8"}

def fetch(url):
    data = b""
    for headers in (AGENT, BROWSER):          # some repositories accept only one of the two identifications
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=45) as r:
                data = r.read(60_000_000)
            if data.startswith(b"%PDF"):
                break
        except Exception:
            continue
    return data

def variants(url):
    host = urllib.parse.urlparse(url).netloc
    if "hal." in host and not url.rstrip("/").endswith(("/document", ".pdf")):
        yield url.rstrip("/") + "/document"
    yield url

def download(urls, target):
    for u in sorted(urls, key=lambda u: not u.lower().endswith((".pdf", "/document"))):
        for v in variants(u):
            try:
                data = fetch(v)
                if not data.startswith(b"%PDF"):
                    link = re.search(r"citation_pdf_url[\"'][^>]*content=[\"']([^\"']+)|content=[\"']([^\"']+)[\"'][^>]*citation_pdf_url", data.decode("utf-8", "ignore"))
                    if not link:
                        continue
                    v = link.group(1) or link.group(2)
                    data = fetch(v)
                if data.startswith(b"%PDF"):
                    target.write_bytes(data)
                    return v
            except Exception:
                continue
    return ""

attempted = 0
for i in lst.sort_values("priority").index:
    target = PDF / f"{i}.pdf"
    if target.exists() or not candidates[i]:
        continue
    if lst.at[i, "download"] == "failed" and lst.at[i, "tried_urls"] == lst.at[i, "urls"] and not RETRY_FAILED:
        continue
    attempted += 1
    origin = download(candidates[i], target)
    lst.at[i, "download"] = "ok" if origin else "failed"
    lst.at[i, "pdf_source"] = origin
    lst.at[i, "tried_urls"] = lst.at[i, "urls"]
    time.sleep(0.5)

lst["pdf"] = ["y" if has_pdf(i) else "n" for i in lst.index]
print("Downloads attempted now:", attempted)
pd.crosstab(lst["priority"], lst["pdf"], margins=True)

Downloads attempted now: 0


pdf,n,y,All
priority,,,
1,69,2,71
2,18,2,20
3,359,18,377
All,446,22,468


## 4. Pending list

In [6]:
ordered = lst.sort_values(["priority", "year"], ascending=[True, False])
columns = ["priority", "reason", "pdf", "year", "citation", "title", "doi", "link", "oa", "oa_status", "urls", "download", "pdf_source", "sources", "tried_urls"]
ordered[columns].to_csv(LIST, index_label="id", encoding="utf-8")

missing_pdf = ordered[(ordered["priority"] <= 2) & (ordered["pdf"] == "n")]
lines = [f"{i}\t{r['link'] or 'no DOI: ' + r['title']}" for i, r in missing_pdf.iterrows()]
Path("priority_pending.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"PDFs available: {(lst['pdf'] == 'y').sum()} of {len(lst)}")
print(f"Priority (1 and 2) without PDF: {len(missing_pdf)} · of which without DOI: {(missing_pdf['doi'] == '').sum()}")
print("With a copy located that has to be fetched by hand (download failed):", int(((lst["download"] == "failed") & (lst["pdf"] == "n")).sum()))
missing_pdf[["priority", "reason", "year", "citation", "link"]].head(25)

PDFs available: 22 of 468
Priority (1 and 2) without PDF: 87 · of which without DOI: 14
With a copy located that has to be fetched by hand (download failed): 34


,priority,reason,year,citation,link
id,,,,,
R1843,1,2 gap criteria,2026,"K. Deng (2026), IEEE Transactions on Very Large Scale Integration (VLSI) Systems",https://doi.org/10.1109/tvlsi.2026.3665483
R1853,1,2 gap criteria,2026,"Ankush (2026), Proceedings of the European Test Workshop",https://doi.org/10.1109/ets69887.2026.11591724
R1835,1,2 gap criteria,2025,"Sunter S. (2025), Proceedings - International Test Conference",https://doi.org/10.1109/itc58126.2025.00006
R1628,1,2 gap criteria,2019,"Li (2019), ACM TRANSACTIONS ON DESIGN AUTOMATION OF ELECTRONIC SYSTEMS",https://doi.org/10.1145/3308566
R2155,1,board level,2019,"Bo O. (2019), 2019 IEEE 2nd International Conference on Electronics and Communication ...",https://doi.org/10.1109/icece48499.2019.9058507
R2138,1,2 gap criteria,2018,"Li Y. (2018), Proceedings of the IEEE VLSI Test Symposium",https://doi.org/10.1109/vts.2018.8368662
R1540,1,2 gap criteria,2017,"Stratigopoulos H.G. (2017), Proceedings of the IEEE VLSI Test Symposium",https://doi.org/10.1109/vts.2017.7928919
R1575,1,board level,2017,"Chen (2017), 2017 PROGNOSTICS AND SYSTEM HEALTH MANAGEMENT CONFERENCE (PHM-HARBIN)",https://doi.org/10.1109/phm.2017.8079296
R1534,1,2 gap criteria,2016,"Huang (2016), IEEE TRANSACTIONS ON COMPUTER-AIDED DESIGN OF INTEGRATED CIRCUITS AND SY...",https://doi.org/10.1109/tcad.2016.2547904


## Conclusions and notes

- **Very little is legally available.** Of 468 included studies, the four sources locate some copy for 56. It is a field published mostly in IEEE, Springer and Elsevier journals and conferences.
- **Downloaded: 22 PDFs.** Among them two of the three G0 candidates (R1436 and R1582) and two works of the control set (R2116 and R0894).
- **34 located copies cannot be downloaded** automatically (the site rejects them or the address no longer exists). They are in the `urls` column to be fetched by hand; five are priority ones.
- **Priority: 87 of 91 are missing**, of which 14 have no DOI and have to be searched by title.
- **By source:** nearly everything retrieved comes from OpenAlex and Semantic Scholar by DOI. HAL by title and the preprint search by title add next to nothing: no version has turned up in Research Square.
- **Route for the rest:** institutional access, interlibrary loan or a request to the author. The PDFs go in `pdf/` named after the identifier, and re-running the notebook updates the list.

### Notes